# Calcula tu inflación — por marca

Este cuaderno calcula la **inflación real por marca** con los datos abiertos de
*Quién es Quién en los Precios* (Profeco), y detecta **reduflación**: cuando el
producto cuesta casi lo mismo pero trae menos contenido.

**No necesitas saber Python ni instalar nada.** Solo ejecuta las celdas en orden.

Para ejecutar una celda: haz clic en ella y presiona el botón ▶ de la izquierda
(o `Shift + Enter`).

---

## Paso 1 — Preparar el cuaderno

Ejecuta la celda de abajo. Tarda unos segundos y descarga el código del análisis.

In [ ]:
#@title Paso 1: preparar (ejecuta esta celda)
!pip install -q pandas
!wget -q -O inflacion_por_marca.py https://raw.githubusercontent.com/carsam68-MonheyB/CALCULA_TU_INFLACI-N/main/inflacion_por_marca.py

import os, glob
os.makedirs("datos", exist_ok=True)
print("Listo. Ya puedes pasar al Paso 2.")

---

## Paso 2 — Subir tus archivos CSV

Tus datos vienen **partidos**: cada mes son varios archivos, numerados al final.
Por ejemplo enero de 2024 son estos dos:

```
01-2024_01.csv     ← enero 2024, parte 1
01-2024_02.csv     ← enero 2024, parte 2
```

Para comparar dos periodos necesitas **todas las partes de un mes de cada año**.
Por ejemplo, para enero 2024 contra enero 2025 subes **4 archivos**:

```
01-2024_01.csv  +  01-2024_02.csv     ← el año viejo
01-2025_01.csv  +  01-2025_02.csv     ← el año nuevo
```

Ejecuta la celda y aparecerá el botón **"Elegir archivos"**. Puedes seleccionar
varios de una vez: mantén presionado `Ctrl` mientras los eliges.

> Si los archivos pesan mucho, la subida tarda varios minutos. Es normal.
> No cierres la pestaña.

In [ ]:
#@title Paso 2: subir los CSV (ejecuta y elige tus archivos)
from google.colab import files
import shutil, os

subidos = files.upload()
for nombre in subidos:
    shutil.move(nombre, os.path.join("datos", nombre))

print("\nArchivos listos en la carpeta datos/:")
for f in sorted(os.listdir("datos")):
    mb = os.path.getsize(os.path.join("datos", f)) / 1e6
    print(f"  {f}  ({mb:,.0f} MB)")

---

## Paso 3 — Decir qué archivos son de cada año

Como cada mes viene partido en varios archivos, aquí no se escribe un nombre
suelto sino un **patrón**: el `*` significa "cualquier cosa", así que
`01-2024_*.csv` agarra la parte 1, la parte 2 y las que haya.

Toma el nombre de tus archivos y cambia el número final por `*`:

| Tus archivos | Lo que escribes |
|---|---|
| `01-2024_01.csv` y `01-2024_02.csv` | `01-2024_*.csv` |
| `08-2025_01.csv` y `08-2025_02.csv` | `08-2025_*.csv` |

- `PATRON_BASE` = el año **viejo** (punto de partida)
- `PATRON_ACTUAL` = el año **nuevo** (con el que comparas)

Usa el **mismo mes** en ambos, para no confundir inflación con temporada.

In [ ]:
#@title Paso 3: indicar los periodos
PATRON_BASE   = "01-2024_*.csv"  #@param {type:"string"}
PATRON_ACTUAL = "01-2025_*.csv"  #@param {type:"string"}

import glob, os
todo_bien = True
for etiqueta, patron in (("BASE", PATRON_BASE), ("ACTUAL", PATRON_ACTUAL)):
    encontrados = sorted(glob.glob(os.path.join("datos", patron)))
    print(f"{etiqueta}:")
    if encontrados:
        for f in encontrados:
            print(f"   {os.path.basename(f)}  ({os.path.getsize(f)/1e6:,.0f} MB)")
    else:
        todo_bien = False
        print("   *** ningun archivo coincide con este patron ***")
    print()

if todo_bien:
    print("Todo listo, pasa al Paso 4.")
else:
    print("Revisa el patron. Archivos que subiste:")
    for f in sorted(os.listdir("datos")):
        print("  ", f)

---

## Paso 4 — Elegir qué analizar

**Importante:** un año de QQP puede traer más de 20 millones de precios. Si no
filtras nada, el cuaderno se queda sin memoria. **Siempre pon al menos un
filtro.**

Escribe los términos separados por espacios. Deja vacío `""` lo que no quieras usar.

Ejemplos:
- `PRODUCTO = "DESODORANTE"` → solo desodorantes
- `PRODUCTO = "DESODORANTE SHAMPOO"` → desodorantes **o** shampoos
- `CATEGORIA = "CUIDADO PERSONAL"` → toda la categoría
- `ESTADO = "COAHUILA"` → solo ese estado

In [ ]:
#@title Paso 4: filtros
PRODUCTO  = "DESODORANTE"  #@param {type:"string"}
MARCA     = ""             #@param {type:"string"}
CATEGORIA = ""             #@param {type:"string"}
ESTADO    = ""             #@param {type:"string"}
CADENA    = ""             #@param {type:"string"}

POR_CADENA = True  #@param {type:"boolean"}
MIN_OBSERVACIONES = 3  #@param {type:"integer"}

if not any((PRODUCTO, MARCA, CATEGORIA, ESTADO, CADENA)):
    print("AVISO: no pusiste ningun filtro. Con archivos grandes esto puede")
    print("       agotar la memoria. Escribe al menos un producto o categoria.")
else:
    print("Filtros listos. Pasa al Paso 5.")

---

## Paso 5 — Calcular

Ejecuta y espera. Con archivos grandes puede tardar varios minutos: verás un
contador de filas mientras avanza.

In [ ]:
#@title Paso 5: calcular (ejecuta esta celda)
import sys, os

cmd = [sys.executable, "inflacion_por_marca.py",
       "--base",   os.path.join("datos", PATRON_BASE),
       "--actual", os.path.join("datos", PATRON_ACTUAL),
       "--min-obs", str(MIN_OBSERVACIONES),
       "--csv", "resultado.csv"]

for bandera, valor in (("--producto", PRODUCTO), ("--marca", MARCA),
                       ("--categoria", CATEGORIA), ("--estado", ESTADO),
                       ("--cadena", CADENA)):
    if valor.strip():
        cmd += [bandera] + valor.split()

if POR_CADENA:
    cmd.append("--por-cadena")

import subprocess
proc = subprocess.run(cmd, capture_output=True, text=True)
print(proc.stdout)
if proc.returncode != 0:
    print("--- detalle ---")
    print(proc.stderr[-3000:])

---

## Paso 6 — Descargar el resultado

Guarda la tabla en tu computadora para abrirla en Excel.

In [ ]:
#@title Paso 6: descargar resultado.csv
from google.colab import files
import os

if os.path.exists("resultado.csv"):
    files.download("resultado.csv")
else:
    print("Todavia no hay resultado.csv. Ejecuta el Paso 5 primero.")

---

## Cómo leer la tabla de reduflación

| Columna | Qué significa |
|---|---|
| **ETIQUETA** | Cuánto subió el precio que ves en el anaquel |
| **CONTENIDO** | Cuánto cambió el tamaño del empaque (negativo = encogió) |
| **REAL** | Cuánto subió el precio por gramo o mililitro |
| **BRECHA** | REAL menos ETIQUETA — la inflación que no se ve |

Una marca con `<-- ENCOGIO` bajó el contenido del empaque. Si su BRECHA es
grande, estás pagando bastante más por gramo aunque el precio del anaquel
casi no se haya movido.

## Notas

- Se usa la **mediana** de precios, no el promedio, para que unos pocos
  registros mal capturados no distorsionen el resultado.
- Un artículo solo aparece si está en **ambos** periodos con al menos
  `MIN_OBSERVACIONES` registros.
- Los archivos que subes se borran cuando cierras la sesión de Colab. Descarga
  siempre tu `resultado.csv`.

Código y documentación: <https://github.com/carsam68-MonheyB/CALCULA_TU_INFLACI-N>